# 27B Parameter QLoRA Fine-Tuning Pipeline
### 4-bit Quantized SFT on Google Gemma 2 27B with `no_robots` in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook demonstrates how to fine-tune a **27 billion parameter foundation model** ([`google/gemma-2-27b-it`](https://huggingface.co/google/gemma-2-27b-it)) on a single GPU using 4-bit QLoRA.

### Pipeline Highlights:
1. **Foundation Model**: `google/gemma-2-27b-it` (Google DeepMind's flagship open-weights model, rivaling 70B models in reasoning and coding).
2. **Curated Dataset**: [`HuggingFaceH4/no_robots`](https://huggingface.co/datasets/HuggingFaceH4/no_robots) (10,000 high-quality, human-written multi-turn instruction conversations).
3. **Memory Optimization**: 4-bit NormalFloat (NF4) quantization shrinks 27B base weights from **54 GB down to ~13.5 GB**, fitting training into **~18–20 GB VRAM**.
4. **Interactive Validation**: Live streaming token generation using `TextStreamer` with full evaluation safeguards (`repetition_penalty=1.15`, KV-caching, and eval mode).


---
## 1. Hardware Requirements & GPU Setup

> [!WARNING]
> **GPU Requirement (24GB+ VRAM Required)**:
> - A 27B parameter model in 4-bit requires **~13.5 GB for weights alone**, and **~18–20 GB total VRAM** during fine-tuning with activation memory and optimizer states.
> - **Google Colab Free Tier (15GB Tesla T4)** is **not sufficient** for 27B training.
> - **Recommended Colab Runtime**: In the top menu, go to **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ select **A100 GPU** (40GB/80GB) or **L4 GPU** (24GB).
> - Also compatible with local GPUs with 24GB+ VRAM (NVIDIA RTX 3090, RTX 4090, A10G, A5000, A6000).


In [ ]:
!nvidia-smi


In [ ]:
# Install core fine-tuning dependencies with pinned compatibility
!pip install -q \
    "torch>=2.4.0" \
    "transformers>=4.45.0,<5.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.11.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=1.0.0" \
    "python-dotenv"


---
## 2. Authentication & Hugging Face Access

`google/gemma-2-27b-it` is an open-weights model, but requires accepting the [Gemma Terms of Use on Hugging Face](https://huggingface.co/google/gemma-2-27b-it) (instant 1-click approval).

> [!IMPORTANT]
> 1. Visit https://huggingface.co/google/gemma-2-27b-it and click **"Acknowledge license"**.
> 2. Pass your Hugging Face Access Token below.
> *(Alternative un-gated 30B model: You can also use `Qwen/Qwen2.5-32B-Instruct` which requires no license click).*


In [ ]:
import os
import getpass

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print("Please enter your Hugging Face Token (required to access Gemma 2):")
    entered_token = getpass.getpass("HF Token: ")
    if entered_token.strip():
        hf_token = entered_token.strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("✓ HF_TOKEN loaded and sanitized.")
else:
    print("⚠️ Warning: No HF_TOKEN provided. If downloading google/gemma-2-27b-it fails, provide your token.")


---
## 3. 27B QLoRA Fine-Tuning Pipeline

### How QLoRA Handles 27 Billion Parameters on a 24GB GPU:
- **Base Weights**: 27.2 billion parameters $\times$ 0.5 bytes (4-bit) = **~13.6 GB**.
- **Double Quantization**: Compresses quantization constants, saving ~350 MB.
- **`bfloat16` Compute Dtype**: Large models with RMSNorm require `bfloat16` to prevent the numerical overflow/underflow seen in FP16.
- **Paged Optimizer (`paged_adamw_8bit`)**: Keeps memory spikes off the GPU by paging optimizer states to CPU RAM.
- **Gradient Checkpointing**: Discards intermediate activations during forward passes and recomputes them during backprop, keeping activation memory under ~3.5 GB.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# Primary 27B Model (or switch to 'Qwen/Qwen2.5-32B-Instruct')
MODEL_ID = "google/gemma-2-27b-it"
DATASET_ID = "HuggingFaceH4/no_robots"
OUTPUT_DIR = "./gemma2-27b-qlora-output"

print(f"1. Loading tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit NormalFloat Quantization Config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # Essential for 27B numerical stability
)

print("2. Loading 27B base model with 4-bit NF4 quantization (this may take 2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Prepare model for k-bit training and disable KV-cache for gradient checkpointing
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# PEFT LoRA Config targeting standard linear attention and MLP projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Load dataset and apply Gemma's official chat template
print(f"3. Loading dataset: {DATASET_ID}...")
raw_dataset = load_dataset(DATASET_ID, split="train")

def format_conversations(sample):
    # Gemma 2 requires strict alternating user/model/user/model turns and rejects 'system'.
    messages = sample["messages"]
    system_prompt = ""
    turns = []

    # 1. Extract system prompt and map assistant -> model
    for msg in messages:
        role = msg.get("role", "").strip()
        content = msg.get("content", "").strip()
        if not content:
            continue
        if role == "system":
            system_prompt += content + "\n\n"
        elif role in ["assistant", "model"]:
            turns.append({"role": "model", "content": content})
        else:  # user
            turns.append({"role": "user", "content": content})

    if not turns:
        return {"text": ""}

    # 2. Attach system prompt to the first user message
    if system_prompt:
        if turns[0]["role"] == "user":
            turns[0]["content"] = system_prompt + turns[0]["content"]
        else:
            turns.insert(0, {"role": "user", "content": system_prompt.strip()})

    # 3. Merge consecutive turns with the same role (e.g. assistant followed by assistant)
    merged_turns = []
    for turn in turns:
        if merged_turns and merged_turns[-1]["role"] == turn["role"]:
            merged_turns[-1]["content"] += "\n\n" + turn["content"]
        else:
            merged_turns.append(turn)

    # 4. Ensure conversation begins with a user message
    if merged_turns and merged_turns[0]["role"] != "user":
        merged_turns.insert(0, {"role": "user", "content": "Hello"})

    # 5. Apply Gemma chat template
    formatted_text = tokenizer.apply_chat_template(
        merged_turns,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted_text}

print("4. Applying chat template to instructions...")
dataset = raw_dataset.map(
    format_conversations,
    remove_columns=raw_dataset.column_names,
    desc="Formatting chat template",
)
print(f"✓ Formatted {len(dataset)} instruction samples ready for training.")


In [ ]:
# Hyperparameters tailored for 27B parameter models
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=1,   # 1 sample per step to keep VRAM <= 20 GB
    gradient_accumulation_steps=16,  # Effective batch size = 1 * 16 = 16
    learning_rate=1e-4,              # Slightly lower LR for 27B to preserve pretrained reasoning
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=True,                       # Native bfloat16 for Ampere/Ada/Hopper architectures
    fp16=False,
    optim="paged_adamw_8bit",        # Automatically offloads memory spikes to system RAM
    gradient_checkpointing=True,     # Recomputes activations on the fly
    report_to="none",
    num_train_epochs=1,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Starting 27B QLoRA fine-tuning...")
trainer.train()

print(f"Saving fine-tuned adapter weights to: {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ Training complete! 27B LoRA adapter weights saved.")


---
## 4. In-Notebook Interactive Evaluation & Testing

Now we evaluate the fine-tuned 27B model directly using pure PyTorch with full generation safeguards:
- **`model.eval()`**: Freezes LoRA dropout to prevent random feature zeroing.
- **`model.config.use_cache = True`**: Restores the KV-cache for fast autoregressive generation.
- **`repetition_penalty = 1.15` & `no_repeat_ngram_size = 3`**: Prevents token loop collapse.
- **`TextStreamer`**: Streams tokens in real time to the Colab cell output.


In [ ]:
from transformers import TextStreamer

# 1. Clean up trainer activation memory
if "trainer" in locals():
    del trainer
torch.cuda.empty_cache()

# 2. Put model in evaluation mode
model.eval()

# 3. Re-enable KV-caching
model.config.use_cache = True

def generate_response(
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
    stream: bool = True,
) -> str:
    """Generates a response from the 27B model using real-time streaming."""
    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None

    # Gemma 2 termination tokens: <end_of_turn> and <eos>
    eos_token_ids = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if isinstance(eot_id, int) and eot_id not in eos_token_ids:
        eos_token_ids.append(eot_id)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_ids,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=3,
            streamer=streamer,
        )

    if not stream:
        generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return ""

test_prompt = "Explain quantum computing in simple terms for a high school student."
print(f"Prompt: {test_prompt}\n")
print("=" * 60)
print("Generated 27B Response (Live Streaming):")
print("=" * 60)
generate_response(test_prompt, max_new_tokens=512, stream=True)


In [ ]:
# Interactive Test: Try your own complex reasoning prompt on the 27B model!
user_prompt = """Write a clean Python script that parses a CSV file containing employee records (name, department, salary) 
and outputs a summary of the average and median salary grouped by department."""

print(f"User Prompt:\n{user_prompt}\n")
print("=" * 60)
print("Generated Response (Live Streaming):")
print("=" * 60)
generate_response(user_prompt, max_new_tokens=512, stream=True)


In [ ]:
import os

print(f"Inspecting saved 27B adapter files in '{OUTPUT_DIR}':\n")
total_size = 0
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        total_size += size_mb
        print(f"  - {fname:<30} ({size_mb:.2f} MB)")

print(f"\nTotal Adapter Size: {total_size:.2f} MB (Notice how compact ~250MB is compared to a 54GB base model!)")


---
## 5. Export or Push Adapter to Hugging Face Hub

Because LoRA trains low-rank decomposition matrices, only the adapter weights (~200–300 MB) need to be exported or pushed to the Hugging Face Hub, not the entire 54 GB base model.


In [ ]:
import shutil

# 1. Package the adapter folder into a zip archive
archive_name = "gemma2_27b_qlora_adapter"
shutil.make_archive(archive_name, 'zip', OUTPUT_DIR)
print(f"✓ Created archive: {archive_name}.zip")

# 2. In Google Colab, uncomment the lines below to download directly:
# from google.colab import files
# files.download(f"{archive_name}.zip")

# 3. Push to Hugging Face Hub (requires write token in HF_TOKEN):
# target_repo = "your-username/gemma-2-27b-it-no-robots-qlora"
# model.push_to_hub(target_repo)
# tokenizer.push_to_hub(target_repo)
# print(f"✓ Successfully published adapter to https://huggingface.co/{target_repo}")


---
## 6. Gemma 2 27B Architectural & Training Reference

### Key Architectural Characteristics
- **Sliding Window Attention (SWA)**: Alternates between local attention (window of 4096 tokens) and global attention (every other layer), significantly reducing KV-cache and attention compute overhead during long sequences.
- **Logit Soft-Capping**: Prevents logit explosion during training without truncating representations:
  $$\text{logits} = \text{cap} \times \tanh\left(\frac{\text{logits}}{\text{cap}}\right)$$
  (Attn logit cap: 50.0, Final logit cap: 30.0).
- **Dual Normalization**: Applies pre-norm and post-norm RMSNorm around every attention and MLP block for training stability at 27B scale.

### Hyperparameter Guide for 27B QLoRA:
| Parameter | Setting | Rationale |
| :--- | :--- | :--- |
| `per_device_train_batch_size` | `1` (or `2` on 80GB) | Keeps peak activation memory within GPU limits. |
| `gradient_accumulation_steps` | `16` (or `8` on 80GB) | Maintains a steady effective batch size of 16. |
| `learning_rate` | `1e-4` | Prevents destabilizing pre-trained world knowledge. |
| `compute_dtype` | `torch.bfloat16` | Required for wide dynamic range; FP16 can overflow. |
| `optim` | `paged_adamw_8bit` | Pages optimizer spikes to host RAM during memory pressure. |

### Troubleshooting: `TemplateError: System role not supported`
- **Cause**: Google Gemma's official chat template strictly accepts only `user` and `model` roles. If a dataset like `no_robots` contains `system` messages, the Jinja template raises `TemplateError: System role not supported`.
- **Solution**: Merge system prompts into the first `user` turn (`system_prompt + content`), and map `assistant` to `model`. This preserves all system instructions while complying with Gemma's tokenizer format.
